# StateGraph Implementation Walkthrough 🤖

## Interactive Presentation with Live Code Demonstrations

**Presenter:** Nandavardhan Doodala  
**Topic:** StateGraph Implementation Walkthrough  
**Training:** GenAI Engineer Training

---

### Opening Statement

Today I will explain how StateGraph works in LangGraph and show a simple implementation using:
- ✅ Shared state
- ✅ Node functions
- ✅ Edges (sequential and conditional)
- ✅ Compilation and execution
- ✅ **Real LLM Integration with Claude Haiku**
- ✅ Retry logic with confidence scoring

By the end of this session, you will understand how to build stateful, flexible AI workflows with REAL LLMs!

---

## 📋 Agenda

1. **Why StateGraph?**
   - Problem with linear chains
   - Need for state, branches, and retries

2. **Core Concepts**
   - State, nodes, and edges
   - `compile()`, `invoke()`, and `stream()`

3. **Implementation Walkthrough** (with live code!)
   - State schema
   - Node functions
   - Graph wiring

4. **Conditional Routing**
   - Confidence-based retry
   - When to end the workflow

5. **Demo and Q&A**
   - Expected output
   - Common interview questions

---

**Takeaway:** By the end, you'll be able to explain and build a basic StateGraph workflow.

---

## 🎯 Slide 3: What is StateGraph?

### Simple Definition

**StateGraph** is a graph-based workflow builder in LangGraph where:
- Each **node** reads the current state and returns updates
- **Edges** define which step runs next
- **State** carries shared data through the workflow

### Why Not Just LangChain Chains?

| Aspect | Chains | StateGraph |
|--------|--------|------------|
| **Flow Type** | Linear only | Linear + Branching |
| **Retries** | Limited | Full control |
| **Shared State** | No | Yes |
| **Debugging** | Hard | Easy |
| **Use Case** | Simple flows | Complex workflows |

### Mental Model

Think of StateGraph like a **workflow diagram in code**:
- 🗂️ **State** = shared memory
- ⚙️ **Nodes** = tasks/functions
- ➡️ **Edges** = control flow

---

## 📊 Slide 4: StateGraph Lifecycle

### 5-Step Process

```
1. Define State Schema
        ↓
2. Create Node Functions
        ↓
3. Connect Edges (sequence or conditional)
        ↓
4. Compile the Graph
        ↓
5. Run with invoke() or stream()
```

### Important Point

StateGraph is a **builder pattern**:
- We **define** the structure using `add_node()` and `add_edge()`
- We **compile** it using `.compile()`
- We **run** it using `.invoke()` or `.stream()`

After compilation, the graph can be invoked, streamed, or run asynchronously.

**Presentation Line:** "The full flow is: define state → add nodes → connect edges → compile → run!"

---

## 💻 DEMO 1: Define Shared State

### Slide 5: Step 1 - Define the Shared State

Let's create a **TypedDict** that defines what data flows through our workflow.

In [ ]:
# Install required packages
import subprocess
import sys

packages = ["langgraph", "langchain", "python-dotenv"]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✅ {package} is already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

print("\n✅ All required packages are ready!")

In [ ]:
# Import required libraries
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

print("✅ All imports successful!")

# ==========================================
# STEP 1: Define the Shared State
# ==========================================

class GraphState(TypedDict):
    """Shared state for our workflow"""
    question: str      # User input
    intent: str        # Classified purpose (technical or general)
    answer: str        # Generated result
    confidence: int    # Quality score (0-100)
    attempts: int      # Retry counter

print("\n" + "="*60)
print("✅ GraphState Schema Defined")
print("="*60)
print("Fields:")
print(f"  - question (str): User input")
print(f"  - intent (str): Classified purpose")
print(f"  - answer (str): Generated result")
print(f"  - confidence (int): Quality score 0-100")
print(f"  - attempts (int): Retry counter")

# Example initial state
initial_state = {
    "question": "What is LangGraph StateGraph?",
    "intent": "",
    "answer": "",
    "confidence": 0,
    "attempts": 0
}

print("\nExample Initial State:")
for key, value in initial_state.items():
    print(f"  {key}: {value}")

In [ ]:
# Initialize Claude Haiku LLM
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
import os

load_dotenv()

# Load API credentials from .env file
api_key = os.getenv("KEY") or "sk-zK7xMXa2pANc64xuf4oaTA"
base_url = os.getenv("BASE_URL") or "https://llmgw-wp.tekstac.com"
model_name = os.getenv("MODEL") or "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize LLM client
llm = ChatAnthropic(
    model=model_name,
    anthropic_api_key=api_key,
    anthropic_api_url=base_url,
    temperature=0.7
)

print("="*60)
print("✅ LLM Initialized Successfully")
print("="*60)
print(f"Model: {model_name}")
print(f"API Endpoint: {base_url}")
print("\n🚀 Ready for real LLM-powered workflow execution!")


### What We Just Did

✅ Defined a **TypedDict** called `GraphState` with 5 fields

**Key Points:**
- State is the **contract** between all nodes
- Each node reads the current state
- Each node returns **only** the fields it updates
- LangGraph automatically merges updates

**Presentation Line:** "State is like a shared whiteboard. Each node writes on it, and the next node reads from it!"

---

## 💻 DEMO 2: Create Node Functions

### Slide 6: Step 2 - Create Node Functions

Now let's create **3 node functions**, each with one clear responsibility.

In [ ]:
# ==========================================
# STEP 2: Create Node Functions (LLM-Powered)
# ==========================================

def classify_intent(state: GraphState) -> dict:
    """
    Node 1: Classify if the question is about LangGraph (technical) or general
    Uses REAL Claude Haiku LLM for classification
    
    Receives: state with 'question' field
    Returns: dict with updated 'intent' field
    """
    question = state["question"]
    
    print(f"\n🔍 [CLASSIFY_INTENT] Analyzing: '{question}'")
    print("   🤖 Calling Claude Haiku LLM...")
    
    # Use real LLM to classify
    prompt = f"""Classify the following question as either 'technical' or 'general'.
    
Question: {question}

Respond with ONLY one word: 'technical' or 'general'"""
    
    response = llm.invoke(prompt)
    intent = response.content.strip().lower()
    
    # Ensure valid intent
    if intent not in ["technical", "general"]:
        intent = "general"
    
    print(f"   ✅ LLM Response: {intent}")
    
    return {"intent": intent}


def generate_answer(state: GraphState) -> dict:
    """
    Node 2: Generate an answer based on classified intent
    Uses REAL Claude Haiku LLM for answer generation
    
    Receives: state with 'question' and 'intent' fields
    Returns: dict with updated 'answer' field
    """
    question = state["question"]
    intent = state["intent"]
    
    print(f"\n✍️  [GENERATE_ANSWER] Intent: {intent}")
    print("   🤖 Calling Claude Haiku LLM...")
    
    # Create system and user messages for LLM
    if intent == "technical":
        system_prompt = """You are an expert in LangGraph and StateGraph. 
Provide a clear, concise explanation to help someone understand the concept."""
    else:
        system_prompt = """You are a helpful assistant. 
The user is asking something outside your expertise area. Politely redirect them."""
    
    # Use real LLM to generate answer
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Question: {question}\n\nProvide a helpful answer in 2-3 sentences.")
    ]
    
    response = llm.invoke(messages)
    answer = response.content.strip()
    
    print(f"   ✅ Generated: {answer[:80]}...")
    
    return {"answer": answer}


def grade_answer(state: GraphState) -> dict:
    """
    Node 3: Grade the answer and assign confidence score
    Uses REAL Claude Haiku LLM for quality assessment
    
    Receives: state with 'answer' and 'intent' fields
    Returns: dict with updated 'confidence' field
    """
    answer = state["answer"]
    intent = state["intent"]
    
    print(f"\n⭐ [GRADE_ANSWER] Assessing answer quality")
    print("   🤖 Calling Claude Haiku LLM for scoring...")
    
    # Use real LLM to score answer quality
    scoring_prompt = f"""Rate the quality of this answer on a scale of 0-100.
Intent: {intent}
Answer: {answer}

Respond with ONLY a number between 0-100 (no explanation)."""
    
    response = llm.invoke(scoring_prompt)
    score_text = response.content.strip()
    
    # Extract numeric score
    try:
        confidence = int(''.join(filter(str.isdigit, score_text)))
        confidence = max(0, min(100, confidence))  # Clamp to 0-100
    except ValueError:
        confidence = 50  # Default if parsing fails
    
    print(f"   ✅ LLM Confidence Score: {confidence}/100")
    
    return {"confidence": confidence}


print("="*60)
print("✅ Node Functions Defined (LLM-Powered)")
print("="*60)
print("1️⃣  classify_intent() - Uses Claude for classification")
print("2️⃣  generate_answer() - Uses Claude for generation")
print("3️⃣  grade_answer() - Uses Claude for quality scoring")
print("\n🚀 All nodes now use REAL LLM calls!")


### What We Just Did

✅ Created **3 node functions**, each:
- Takes `state: GraphState` as input
- Performs one clear task
- Returns `dict` with **only** the fields it updates

**Key Points:**
- Each node has **one responsibility** (Single Responsibility Principle)
- Nodes are **reusable** in other workflows
- Nodes are **testable** independently
- Print statements help with **debugging**

**Presentation Line:** "Notice that each node only returns the fields it changes. LangGraph merges these updates automatically!"

---

## 💻 DEMO 3: Wire the Graph

### Slide 7: Step 3 - Wire the Graph and Run

Now let's **connect nodes with edges** and create our first workflow!

In [ ]:
# ==========================================
# STEP 3: Build and Wire the Graph
# ==========================================

# Create the StateGraph builder
builder = StateGraph(GraphState)

# Add nodes (node_name, function)
builder.add_node("classify", classify_intent)
builder.add_node("generate", generate_answer)
builder.add_node("grade", grade_answer)

print("✅ Nodes added to graph")
print("   - classify")
print("   - generate")
print("   - grade")

# Connect edges (linear flow)
builder.add_edge(START, "classify")      # Start → classify
builder.add_edge("classify", "generate") # classify → generate
builder.add_edge("generate", "grade")    # generate → grade
builder.add_edge("grade", END)           # grade → End

print("\n✅ Edges connected")
print("   START → classify → generate → grade → END")

# Compile the graph
app = builder.compile()
print("\n✅ Graph compiled successfully!")

In [ ]:
# ==========================================
# RUN THE GRAPH
# ==========================================

print("\n" + "="*60)
print("🚀 RUNNING WORKFLOW: Example 1 - Technical Question")
print("="*60)

# Initial state with a technical question
initial_state_1 = {
    "question": "What is LangGraph StateGraph?",
    "intent": "",
    "answer": "",
    "confidence": 0,
    "attempts": 0
}

# Invoke the graph
result_1 = app.invoke(initial_state_1)

print("\n" + "="*60)
print("📊 FINAL RESULT")
print("="*60)
print(f"Question: {result_1['question']}")
print(f"Intent: {result_1['intent']}")
print(f"Answer: {result_1['answer']}")
print(f"Confidence: {result_1['confidence']}/100")
print(f"Attempts: {result_1['attempts']}")

In [ ]:
# Test with another example
print("\n" + "="*60)
print("🚀 RUNNING WORKFLOW: Example 2 - General Question")
print("="*60)

initial_state_2 = {
    "question": "What is the weather today?",
    "intent": "",
    "answer": "",
    "confidence": 0,
    "attempts": 0
}

result_2 = app.invoke(initial_state_2)

print("\n" + "="*60)
print("📊 FINAL RESULT")
print("="*60)
print(f"Question: {result_2['question']}")
print(f"Intent: {result_2['intent']}")
print(f"Answer: {result_2['answer']}")
print(f"Confidence: {result_2['confidence']}/100")
print(f"Attempts: {result_2['attempts']}")

### What We Just Did

✅ **Built and executed our first workflow!**

**Graph Structure:**
```
START
  ↓
classify_intent()
  ↓
generate_answer()
  ↓
grade_answer()
  ↓
END
```

**Key Observations:**
1. State flows through each node sequentially
2. Each node updates specific fields
3. Output contains complete state after all nodes execute
4. Different inputs produce different outputs (classification-based)

**Presentation Line:** "This is a simple linear workflow. Now let's make it smarter with conditional routing and retry logic!"

---

## 💻 DEMO 4: Add Conditional Routing & Retry Logic

### Slide 8: Step 4 - Add Conditional Routing and Retry

Now let's upgrade our workflow with **smart routing** and **automatic retry logic**!

In [ ]:
# ==========================================
# STEP 4: Add Conditional Routing
# ==========================================

def route_by_confidence(state: GraphState) -> Literal["retry", "end"]:
    """
    Router function: Decide whether to retry or end based on confidence and attempts.
    
    Returns:
    - "retry": if confidence is low AND attempts < 2
    - "end": if confidence is high OR max attempts reached
    """
    confidence = state["confidence"]
    attempts = state["attempts"]
    
    print(f"\n🔄 [ROUTER] Evaluating:")
    print(f"   Confidence: {confidence}/100")
    print(f"   Attempts: {attempts}/2")
    
    # Decision logic
    if confidence >= 70:
        print(f"   Decision: ✅ END (high confidence)")
        return "end"
    
    if attempts >= 2:
        print(f"   Decision: ✅ END (max attempts reached)")
        return "end"
    
    print(f"   Decision: 🔄 RETRY (low confidence + attempts available)")
    return "retry"


def retry_answer(state: GraphState) -> dict:
    """
    Node: Increment retry counter and regenerate answer with different approach
    """
    new_attempts = state["attempts"] + 1
    
    # For demo, modify the question to trigger different classification
    new_question = state["question"] + " (retry attempt #" + str(new_attempts) + ")"
    
    print(f"\n🔄 [RETRY] Attempting to improve answer (attempt #{new_attempts})")
    
    return {"attempts": new_attempts, "question": new_question}


print("✅ Router and retry functions defined")

In [ ]:
# ==========================================
# BUILD NEW GRAPH WITH CONDITIONAL ROUTING
# ==========================================

# Create a new StateGraph with conditional routing
builder_advanced = StateGraph(GraphState)

# Add all nodes
builder_advanced.add_node("classify", classify_intent)
builder_advanced.add_node("generate", generate_answer)
builder_advanced.add_node("grade", grade_answer)
builder_advanced.add_node("retry", retry_answer)

print("✅ All nodes added (including retry)")

# Linear edges from START to grade
builder_advanced.add_edge(START, "classify")
builder_advanced.add_edge("classify", "generate")
builder_advanced.add_edge("generate", "grade")

# Conditional edges from grade
builder_advanced.add_conditional_edges(
    "grade",
    route_by_confidence,
    {
        "retry": "retry",  # If retry, go to retry node
        "end": END         # If end, terminate
    }
)

# From retry, loop back to generate
builder_advanced.add_edge("retry", "generate")

print("✅ Conditional edges added")

# Compile
app_advanced = builder_advanced.compile()
print("\n✅ Advanced graph compiled with conditional routing!")

print("\nGraph Flow:")
print("  START → classify → generate → grade")
print("                       ↑         │")
print("                       └─ retry ←┘ (if confidence < 70 and attempts < 2)")
print("                       END (if confidence ≥ 70 or attempts = 2)")

In [ ]:
# ==========================================
# TEST CONDITIONAL ROUTING
# ==========================================

print("\n" + "="*60)
print("🚀 RUNNING ADVANCED WORKFLOW: With Retry Logic")
print("="*60)

# Test with a technical question (high confidence)
print("\n📝 Test Case 1: Technical Question (Should NOT retry)")
result_adv_1 = app_advanced.invoke({
    "question": "How do I use StateGraph?",
    "intent": "",
    "answer": "",
    "confidence": 0,
    "attempts": 0
})

print(f"\n✅ Final State:")
print(f"   Question: {result_adv_1['question']}")
print(f"   Intent: {result_adv_1['intent']}")
print(f"   Confidence: {result_adv_1['confidence']}/100")
print(f"   Attempts: {result_adv_1['attempts']}")
print(f"   Answer: {result_adv_1['answer'][:60]}...")

In [ ]:
# Test with a general question (low confidence - will trigger retry)
print("\n" + "="*60)
print("📝 Test Case 2: General Question (Should trigger retry logic)")
print("="*60)

result_adv_2 = app_advanced.invoke({
    "question": "What is the capital of France?",
    "intent": "",
    "answer": "",
    "confidence": 0,
    "attempts": 0
})

print(f"\n✅ Final State After Retries:")
print(f"   Question: {result_adv_2['question']}")
print(f"   Intent: {result_adv_2['intent']}")
print(f"   Confidence: {result_adv_2['confidence']}/100")
print(f"   Attempts: {result_adv_2['attempts']}")
print(f"   Answer: {result_adv_2['answer'][:60]}...")
print(f"\n   Note: This question has low confidence (40), so it triggered retries.")
print(f"   After {result_adv_2['attempts']} attempt(s), the workflow ended.")

### What We Just Did

✅ **Added smart conditional routing and retry logic!**

**Key Concepts:**

1. **Router Function** - Returns `Literal["retry", "end"]`
   - Decides next step based on current state
   - Type-safe with Literal hints

2. **Conditional Edges** - `add_conditional_edges(from, router_fn, {path: node})`
   - Routes to different nodes based on logic
   - Enables dynamic workflows

3. **Retry Loop** - `retry_node → generate_node`
   - Allows multiple attempts to improve answer
   - Max attempts prevent infinite loops

**Advantages:**
- ✅ Automatic retry on low confidence
- ✅ Prevents infinite loops
- ✅ Full visibility into routing decisions
- ✅ Easy to debug and test

**Presentation Line:** "This is where StateGraph becomes powerful! We have branching logic, retries, and full control over execution!"

---

## 🏢 Slide 9: Where StateGraph is Useful

### Real-World Use Cases

### 1. **RAG Pipelines** (Retrieval-Augmented Generation)
```
User Query
    ↓
Retrieve Documents
    ↓
Generate Answer
    ↓
Verify Groundedness ←─ If not grounded, refine
    ↓
Return Answer
```

### 2. **Agent Workflows**
```
User Request
    ↓
Planner (break into steps)
    ↓
Tool Execution (use external APIs)
    ↓
Validator (verify results) ←─ If invalid, retry
    ↓
Human Approval (for high-impact decisions)
    ↓
Return Result
```

### 3. **Enterprise Automation**
```
Ticket/Request
    ↓
Classify (priority, category)
    ↓
Route (assign team)
    ↓
Process (execute action)
    ↓
Review & Approval ←─ If rejected, reassign
    ↓
Close
```

### Main Value

StateGraph is useful wherever workflows need:
- 🔀 **Decision points** (branching logic)
- ♻️ **Retries** (automatic recovery)
- 📊 **Validation** (quality checks)
- 👁️ **Observability** (tracking and debugging)
- 🔒 **Control** (explicit flow definition)

**Interview Question:** "When would you use StateGraph instead of a simple LangChain chain?"

**Answer:** "When your workflow needs branching, retries, validation, or shared state across multiple steps. StateGraph gives explicit control and visibility that chains don't provide."

---

## 🎓 Key Takeaways and Q&A

### Slide 10: Key Takeaways

#### 1. **StateGraph in One Line**
> A workflow graph where nodes update shared state and edges decide the next step.

#### 2. **Why Compile?**
- `compile()` transforms the builder into an executable graph
- Enables optimization and validation
- Turns the graph definition into a runnable application

#### 3. **What Does a Node Return?**
- A node returns a **partial update** to the state
- Only include fields that change
- LangGraph merges updates automatically

#### 4. **What is Conditional Routing?**
- Choosing the next node based on values in the current state
- Implemented via router functions that return `Literal[...]`
- Enables dynamic, adaptive workflows

#### 5. **Best Practice**
- ✅ Keep state explicit and simple
- ✅ Keep nodes small and focused
- ✅ Use meaningful names for nodes and edges
- ✅ Test nodes independently before combining
- ✅ Use conditional routing for complex flows

### Closing

**Presentation Line:**

> To summarize: StateGraph helps us build stateful, controlled, and flexible AI workflows using shared state, modular nodes, and routing logic. You now understand the core concepts and can build your own workflows!

**Thank you!**

---

## 🤔 Common Interview Questions

### Q1: What's the difference between StateGraph and LangChain chains?
**A:** Chains are linear, StateGraph supports branching, retries, and shared state. StateGraph is better for complex workflows.

### Q2: Why return a partial dict instead of the full state?
**A:** LangGraph merges updates automatically. Returning partial state makes nodes reusable and testable.

### Q3: What happens if a router function returns an invalid path?
**A:** LangGraph will raise an error. Always ensure router returns one of the defined Literal values.

### Q4: Can nodes be executed in parallel?
**A:** Yes, if they don't depend on each other. You can use `add_edge()` to create parallel branches.

### Q5: How do you handle errors in StateGraph?
**A:** You can catch errors in nodes and return an error state, or use conditional routing to handle edge cases.

### Q6: What's the difference between `invoke()` and `stream()`?
**A:** `invoke()` runs the graph to completion and returns the final state. `stream()` yields intermediate states as each node completes.

### Q7: Can you use StateGraph in production?
**A:** Yes! Add checkpointers for state persistence, use async methods, and deploy with FastAPI or similar frameworks.